In [1]:
%%capture --no-stderr
%pip install --upgrade --force-reinstall langgraph
%pip install --quiet -U langchain_openai langchain_core
%pip install requests

In [2]:
%pip install presidio_analyzer
%pip install presidio_anonymizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.7/128.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 2.0 MB/s eta 0:00:00


In [3]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Define tools and bind to LLM

In [12]:
from langchain_openai import ChatOpenAI
from google.colab import userdata
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig


def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second intfrom langchain_openai import ChatOpenAI
    """
    return a * b

# This will be a tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a and b.

    Args:
        a: first int
        b: second int
    """
    return a / b

tools = [add, multiply, divide]

llm = ChatOpenAI(
    api_key=userdata.get('OPENAI_API_KEY'),
    model="gpt-4o")

results = llm.invoke([
    ('system', 'Please assist the user in calculating'),
    ('user', 'What is 234293874 / 342, think step-by-step')
])

print(results)

llm_with_tools = llm.bind_tools(tools)

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# Node
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

## Build the graph

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display

# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "assistant")

checkpointer = InMemorySaver()


react_graph = builder.compile(debug=True, checkpointer=checkpointer)

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

## Invoke the agent

In [4]:
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry

registry = RecognizerRegistry()
registry.load_predefined_recognizers()

# Add the recognizer to the existing list of recognizers
# registry.add_recognizer(titles_recognizer)

# Set up analyzer with our updated recognizer registry
analyzer = AnalyzerEngine(registry=registry)

user_msg = "My name is Pratik. my phone number is 217-299-4532"

results = analyzer.analyze(text=user_msg,
                            language='en')

results

[type: PHONE_NUMBER, start: 38, end: 50, score: 0.75]

In [ ]:
user_msg = "My name is Pratik. my phone number is 217-299-4532"

results = analyzer.analyzer(text=user_msg,
                            language='en')

for result in results:
  print(result)
  user_msg[result.start]


messages = [HumanMessage(content="Add 2232 and 222222, then divide the whole thing by 14.")]
messages = react_graph.invoke({"messages": messages}, config)
for m in messages['messages']:
    m.pretty_print()

[values] {'messages': [HumanMessage(content='Add 2232 and 222222, then divide the whole thing by 14.', additional_kwargs={}, response_metadata={}, id='3254c4cd-4f19-41f1-83b2-c4015dfd3f6b')]}
[updates] {'assistant': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 152, 'total_tokens': 205, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_b1442291a8', 'id': 'chatcmpl-CbXaUeP2Kduj7McQt79NRpdPdZrOh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--e987d8b8-7640-4b19-a363-64c5e7ef205c-0', tool_calls=[{'name': 'add', 'args': {'a': 2232, 'b': 222222}, 'id': 'call_JXhxi52bAypoDUIdAjipPPzo', 'type': 'tool_call'},

In [ ]:
config = {"configurable": {"thread_id": "1"}}
react_graph.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='Add 389573 and 4439374, then divide the whole thing by 7.', additional_kwargs={}, response_metadata={}, id='399764c0-ccd2-4de7-8015-6ab342d8f7fa'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 153, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_b1442291a8', 'id': 'chatcmpl-CbXWgUGz5BR5QeCwawqr97rdYvgmD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--d7e41505-bd60-45ee-aa93-35b2f9c840f2-0', tool_calls=[{'name': 'add', 'args': {'a': 389573, 'b': 4439374}, 'id': 'call_KoevFplr2PP71pWIsVqJWwiL', 'type': 'tool_call'}, {'name': 'divide', 'a